# Generation of 3D Structures of Organocatalysts for an Asymmetric Mannich Reaction Dataset

This notebook accompanies the dataset paper. It generates 3D conformers of all unique organocatalysts present in the dataset of asymmetric Mannich reactions and exports them as MOL V3000 SDF files together with an enriched data table containing the activation free energy difference (ΔΔG‡) calculated from the experimental enantiomeric excess.

## Pipeline overview

1. Read the curated reaction table (CSV or XLSX).
2. Compute ΔΔG‡ from `ee` and `temperature` via the Eyring relation.
3. For each unique catalyst, build a 3D structure using one of three strategies depending on the type of chirality (point, free atropoisomeric, or bridged atropoisomeric).
4. Validate each structure (canonical SMILES match, biaryl dihedral sign for atropoisomers, stereocenter count).
5. Diagnose enantiomeric R/S pairs by computing direct and mirror-aligned RMSD.
6. Export the enriched dataset and an archive of SDF files suitable for downstream descriptor computation (Sterimol, %Vbur, MORFEUS, pmapper) and machine-learning workflows.

## Reproducibility

All conformational sampling uses ETKDGv3 with a fixed random seed. Re-running the notebook on the same input file with the same RDKit version produces byte-identical SDF outputs.

## 1. Dependencies

The notebook relies on RDKit for cheminformatics, pandas for tabular data, and tqdm for progress reporting. Force-field optimization uses MMFF94 as implemented in RDKit.

In [ ]:
# Uncomment to install if needed:
# !pip install rdkit pandas tqdm openpyxl

In [ ]:
import os
import re
import shutil
import warnings
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Lipinski, rdMolAlign, rdMolTransforms

RDLogger.DisableLog("rdApp.warning")
RDLogger.DisableLog("rdApp.error")
warnings.filterwarnings("ignore")

print(f"RDKit version: {Chem.rdBase.rdkitVersion}")
print(f"Pandas version: {pd.__version__}")
print(f"Timestamp:      {datetime.now().isoformat(timespec='seconds')}")

## 2. Configuration

All user-tunable parameters are grouped here. The remainder of the notebook depends only on these variables.

In [ ]:
# Input/output paths
INPUT_FILE     = "mannich_dataset.xlsx"   # path to curated reaction table (.csv or .xlsx)
INPUT_SHEET    = 0                        # sheet name or index (xlsx only)
OUTPUT_SDF_DIR = "catalysts_3d"           # directory for individual SDF files
OUTPUT_ARCHIVE = "catalysts_3d.zip"       # ZIP archive of SDF files + metadata

# Column names in the input table
COL_SMILES      = "organocatalyst"
COL_AXIAL       = "axial_configuration"
COL_EE          = "ee"
COL_TEMPERATURE = "temperature"

# Output naming
CATALYST_PREFIX = "catalyst"              # catalyst1.sdf, catalyst2.sdf, ...

# 3D generation parameters
RANDOM_SEED      = 42
N_CONFORMERS     = 5                      # ETKDGv3 conformers for point-chiral and free biaryl
N_BRIDGED_SEEDS  = 20                     # seeds tested for bridged-biaryl atropoisomers
MAX_OPT_ITERS    = 500                    # MMFF94 optimization iterations

# ΔΔG calculation
TEMPERATURE_UNIT = "celsius"              # "celsius" or "kelvin"
EE_UPPER_CAP_PCT = 99.9                   # cap for ee = 100% to avoid log(infinity)
COMPUTE_DDG      = True

# Pre-flight validation
DO_PREFLIGHT_CHECK = True

# Derived paths (auto)
_input_path = Path(INPUT_FILE)
_ext        = _input_path.suffix.lower()
assert _ext in (".csv", ".xlsx", ".xls"), f"Unsupported input format: {_ext}"
INPUT_FORMAT = "excel" if _ext in (".xlsx", ".xls") else "csv"
OUTPUT_ENRICHED_FULL    = f"{_input_path.stem}_full{_ext}"
OUTPUT_ENRICHED_MINIMAL = f"{_input_path.stem}_minimal{_ext}"
CATALYST_MAPPING_CSV    = "catalyst_mapping.csv"

Path(OUTPUT_SDF_DIR).mkdir(parents=True, exist_ok=True)

print(f"Input file:        {INPUT_FILE}")
print(f"Input format:      {INPUT_FORMAT}")
print(f"SDF directory:     {Path(OUTPUT_SDF_DIR).resolve()}")
print(f"Output (full):     {OUTPUT_ENRICHED_FULL}")
print(f"Output (minimal):  {OUTPUT_ENRICHED_MINIMAL}")

## 3. Core functions for 3D structure generation

Organocatalysts in the dataset fall into three chirality categories, each requiring a distinct embedding strategy:

| Strategy | Catalyst class | 3D generation |
|---|---|---|
| `plain` | Prolines, MacMillan imidazolidinones, cinchona alkaloids, thioureas | Standard ETKDGv3 embedding with `enforceChirality=True`; stereo encoded in SMILES via `@`/`@@`. |
| `free_biaryl` | Classical BINOL, SPINOL, TRIP phosphoric acids | Embedding followed by setting the biaryl dihedral to ±90° and minimization with an MMFF torsion constraint (force constant 10⁴, range ±5°). |
| `bridged_biaryl` | NOBIN derivatives where the biaryl axis is part of a ring | Multiple ETKDGv3 seeds (`N_BRIDGED_SEEDS`); conformers with the correct dihedral sign are retained, and the lowest-energy MMFF94 conformer is selected. |

The strategy is determined automatically by examining the molecular graph and the value of `axial_configuration`.

In [ ]:
def _is_filled(value) -> bool:
    """Return True if a cell value is meaningfully populated."""
    if value is None:
        return False
    if isinstance(value, float) and np.isnan(value):
        return False
    if isinstance(value, str) and value.strip().lower() in ("", "nan", "none", "na"):
        return False
    return True


def find_biaryl_axis(mol):
    """Locate a biaryl single bond suitable for an atropoisomeric axis.

    Returns a tuple ``(atom_idx_1, atom_idx_2)`` or ``None`` if no such bond
    is present. Both atoms must be aromatic carbons in distinct aromatic rings,
    and both ortho positions must be substituted (creating a rotation barrier).
    """
    ri = mol.GetRingInfo()
    aromatic_rings = [r for r in ri.AtomRings()
                      if all(mol.GetAtomWithIdx(idx).GetIsAromatic() for idx in r)]
    for bond in mol.GetBonds():
        if bond.GetBondType() != Chem.BondType.SINGLE or bond.GetIsAromatic():
            continue
        a1, a2 = bond.GetBeginAtom(), bond.GetEndAtom()
        if not (a1.GetIsAromatic() and a2.GetIsAromatic()):
            continue
        if a1.GetSymbol() != "C" or a2.GetSymbol() != "C":
            continue
        rings_a1 = {tuple(r) for r in aromatic_rings if a1.GetIdx() in r}
        rings_a2 = {tuple(r) for r in aromatic_rings if a2.GetIdx() in r}
        if rings_a1 & rings_a2 or not rings_a1 or not rings_a2:
            continue

        def _ortho_substituted(atom, partner_idx):
            for n in atom.GetNeighbors():
                if n.GetIdx() == partner_idx or not n.GetIsAromatic():
                    continue
                for sub in n.GetNeighbors():
                    if sub.GetIdx() in (atom.GetIdx(), partner_idx):
                        continue
                    if sub.GetSymbol() != "H":
                        return True
            return False

        if _ortho_substituted(a1, a2.GetIdx()) and _ortho_substituted(a2, a1.GetIdx()):
            return (a1.GetIdx(), a2.GetIdx())
    return None


def _ortho_neighbor(mol, atom_idx, partner_idx):
    """Return the index of an aromatic ortho neighbor of ``atom_idx``."""
    for n in mol.GetAtomWithIdx(atom_idx).GetNeighbors():
        if n.GetIdx() != partner_idx and n.GetIsAromatic():
            return n.GetIdx()
    return None


def _bond_in_ring(mol, idx1, idx2):
    """Return True if the bond between two atoms is part of a ring."""
    bond = mol.GetBondBetweenAtoms(idx1, idx2)
    return bond.IsInRing() if bond is not None else False


def _measure_dihedral(mol, axis, conf_id=-1):
    """Measure the biaryl dihedral angle for a given axis."""
    a1, a2 = axis
    o1 = _ortho_neighbor(mol, a1, a2)
    o2 = _ortho_neighbor(mol, a2, a1)
    if o1 is None or o2 is None:
        return None
    conf = mol.GetConformer(conf_id) if conf_id >= 0 else mol.GetConformer()
    return rdMolTransforms.GetDihedralDeg(conf, o1, a1, a2, o2)


def _mmff_energy(mol, conf_id=-1):
    """Return MMFF94 energy of a conformer, or infinity on failure."""
    try:
        mp = AllChem.MMFFGetMoleculeProperties(mol)
        ff = AllChem.MMFFGetMoleculeForceField(mol, mp, confId=conf_id)
        return ff.CalcEnergy() if ff else np.inf
    except Exception:
        return np.inf


def _embed(mol, seed, use_random_coords=False):
    """Embed a single conformer with ETKDGv3."""
    params = AllChem.ETKDGv3()
    params.randomSeed = seed
    params.useSmallRingTorsions = True
    params.enforceChirality = True
    params.useRandomCoords = use_random_coords
    return AllChem.EmbedMolecule(mol, params)

In [ ]:
def build_3d_catalyst(smiles, axial_config="none",
                     n_conformers=N_CONFORMERS,
                     n_bridged_seeds=N_BRIDGED_SEEDS,
                     random_seed=RANDOM_SEED):
    """Generate a 3D conformer for an organocatalyst.

    The strategy is selected automatically:

    * ``plain`` — standard embedding for catalysts without a biaryl axis.
    * ``free_biaryl`` — embedding plus torsion constraint to enforce
      the requested axial configuration for free biaryls.
    * ``bridged_biaryl`` — multiple ETKDGv3 seeds for biaryl axes inside
      a ring, retaining conformers with the correct dihedral sign and
      returning the lowest-energy one.

    Returns
    -------
    mol : RDKit Mol or None
        Molecule with a single 3D conformer (with hydrogens), or None on failure.
    source : str
        Strategy label (``"smiles"``, ``"smiles+axial_free"``,
        ``"smiles+axial_bridged"``, ``"smiles_no_axis_applied"``)
        or an error code.
    info : dict
        Diagnostic data including ``axis``, ``strategy``, ``final_angle``,
        and ``n_valid_conformers`` (for the bridged strategy).
    """
    if not _is_filled(smiles):
        return None, "no_smiles", {}

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, "invalid_smiles", {}

    mol = Chem.AddHs(mol)

    axis = find_biaryl_axis(mol)
    needs_axial = _is_filled(axial_config) and str(axial_config).strip().upper() in ("R", "S")
    config = str(axial_config).strip().upper() if needs_axial else None

    if not needs_axial:
        strategy = "plain"
    elif axis is None:
        strategy = "plain_no_axis"
    else:
        strategy = "bridged_biaryl" if _bond_in_ring(mol, axis[0], axis[1]) else "free_biaryl"

    info = {"axis": axis, "strategy": strategy}

    # --- Strategy 1: bridged biaryl (NOBIN-type) ---
    if strategy == "bridged_biaryl":
        target_sign = +1 if config == "R" else -1
        candidates = []

        for seed in range(random_seed, random_seed + n_bridged_seeds):
            mol_try = Chem.AddHs(Chem.MolFromSmiles(smiles))
            rc = _embed(mol_try, seed)
            if rc < 0:
                rc = _embed(mol_try, seed, use_random_coords=True)
            if rc < 0:
                continue
            try:
                AllChem.MMFFOptimizeMolecule(mol_try, maxIters=MAX_OPT_ITERS)
            except Exception:
                continue

            angle = _measure_dihedral(mol_try, axis)
            if angle is None or np.sign(angle) != target_sign:
                continue
            candidates.append((_mmff_energy(mol_try), mol_try, angle))

        if not candidates:
            return None, f"bridged_no_{config}_conformer", info

        candidates.sort(key=lambda x: x[0])
        best_mol = candidates[0][1]
        info["final_angle"] = candidates[0][2]
        info["n_valid_conformers"] = len(candidates)
        Chem.AssignStereochemistryFrom3D(best_mol)
        return best_mol, "smiles+axial_bridged", info

    # --- Strategy 2: free biaryl (classical BINOL) ---
    if strategy == "free_biaryl":
        a1, a2 = axis
        o1 = _ortho_neighbor(mol, a1, a2)
        o2 = _ortho_neighbor(mol, a2, a1)

        params = AllChem.ETKDGv3()
        params.randomSeed = random_seed
        params.useSmallRingTorsions = True
        params.enforceChirality = True
        conf_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_conformers, params=params))

        if not conf_ids:
            for sd in range(random_seed + 100, random_seed + 110):
                rc = _embed(mol, sd, use_random_coords=True)
                if rc >= 0:
                    conf_ids = [rc]
                    break

        if not conf_ids:
            return None, "embed_failed", info

        target = 90.0 if config == "R" else -90.0
        for cid in conf_ids:
            try:
                rdMolTransforms.SetDihedralDeg(mol.GetConformer(cid), o1, a1, a2, o2, target)
                mp = AllChem.MMFFGetMoleculeProperties(mol)
                ff = AllChem.MMFFGetMoleculeForceField(mol, mp, confId=cid)
                if ff is not None:
                    lo, hi = (85.0, 95.0) if config == "R" else (-95.0, -85.0)
                    ff.MMFFAddTorsionConstraint(o1, a1, a2, o2, False, lo, hi, 1.0e4)
                    ff.Minimize(maxIts=MAX_OPT_ITERS)
                else:
                    AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=MAX_OPT_ITERS)
            except Exception:
                pass

        best_cid = min(conf_ids, key=lambda cid: _mmff_energy(mol, cid))
        best_mol = Chem.Mol(mol)
        best_mol.RemoveAllConformers()
        best_mol.AddConformer(mol.GetConformer(best_cid), assignId=True)
        Chem.AssignStereochemistryFrom3D(best_mol)
        info["final_angle"] = _measure_dihedral(best_mol, axis)
        return best_mol, "smiles+axial_free", info

    # --- Strategy 3: plain (point chirality from SMILES) ---
    params = AllChem.ETKDGv3()
    params.randomSeed = random_seed
    params.useSmallRingTorsions = True
    params.enforceChirality = True
    conf_ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n_conformers, params=params))

    if not conf_ids:
        for sd in range(random_seed + 100, random_seed + 110):
            rc = _embed(mol, sd, use_random_coords=True)
            if rc >= 0:
                conf_ids = [rc]
                break

    if not conf_ids:
        return None, "embed_failed", info

    for cid in conf_ids:
        try:
            AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=MAX_OPT_ITERS)
        except Exception:
            pass

    best_cid = min(conf_ids, key=lambda cid: _mmff_energy(mol, cid))
    best_mol = Chem.Mol(mol)
    best_mol.RemoveAllConformers()
    best_mol.AddConformer(mol.GetConformer(best_cid), assignId=True)
    Chem.AssignStereochemistryFrom3D(best_mol)
    if axis is not None:
        info["final_angle"] = _measure_dihedral(best_mol, axis)

    source = "smiles_no_axis_applied" if strategy == "plain_no_axis" else "smiles"
    return best_mol, source, info

In [ ]:
def validate_3d_structure(mol, expected_axial=None, expected_smiles=None, info=None):
    """Validate a generated 3D structure against the input specification.

    Checks:

    * ``structure_match`` — connectivity of the built molecule matches the SMILES.
    * ``axial_match`` — biaryl dihedral has the correct sign for atropoisomers.
    * ``axial_angle`` — measured dihedral in degrees.
    * ``n_point_stereocenters`` — number of point stereocenters identified from 3D.
    """
    checks = {}

    if expected_smiles:
        try:
            canon_built = Chem.MolToSmiles(Chem.RemoveHs(mol), isomericSmiles=False)
            canon_expect = Chem.MolToSmiles(Chem.MolFromSmiles(expected_smiles), isomericSmiles=False)
            checks["structure_match"] = (canon_built == canon_expect)
        except Exception:
            checks["structure_match"] = False

    if expected_axial and str(expected_axial).strip().upper() in ("R", "S"):
        axis = info.get("axis") if info else find_biaryl_axis(mol)
        if axis is None:
            checks["axial_match"] = None
            checks["axial_angle"] = None
        else:
            angle = _measure_dihedral(mol, axis)
            expected_sign = +1 if str(expected_axial).strip().upper() == "R" else -1
            checks["axial_match"] = (
                bool(np.sign(angle) == expected_sign) if angle is not None else None
            )
            checks["axial_angle"] = float(angle) if angle is not None else None

    no_h = Chem.RemoveHs(mol)
    Chem.AssignStereochemistryFrom3D(no_h)
    centers = Chem.FindMolChiralCenters(no_h, includeUnassigned=False,
                                         useLegacyImplementation=False)
    checks["n_point_stereocenters"] = len(centers)
    return checks

## 4. Load and prepare the dataset

The reaction table is read into a pandas DataFrame. Decimal values that use a comma separator (a frequent issue in spreadsheets from non-English locales) are converted to standard decimal points. Each unique catalyst is assigned a `catalyst_id` based on its full InChIKey and axial configuration, ensuring that stereoisomers of the same connectivity (e.g., L- and D-proline) receive distinct identifiers.

In [ ]:
# Load the dataset
if INPUT_FORMAT == "excel":
    df = pd.read_excel(INPUT_FILE, sheet_name=INPUT_SHEET)
else:
    df = pd.read_csv(INPUT_FILE)

print(f"Rows loaded: {len(df)}")
print(f"Columns:     {list(df.columns)}")

# Validate required columns
missing = [c for c in [COL_SMILES, COL_AXIAL] if c not in df.columns]
assert not missing, f"Required columns are missing from input: {missing}"


# Convert comma decimals ("1,5" -> 1.5) where applicable
def _fix_comma_decimal(series):
    """Convert decimal commas to dots; leave non-numeric strings untouched."""
    if pd.api.types.is_numeric_dtype(series):
        return series, False

    decimal_comma_re = re.compile(r"^-?\d+,\d+$")
    has_comma_decimal = any(
        decimal_comma_re.match(str(v).strip())
        for v in series.dropna()
    )
    if not has_comma_decimal:
        return series, False

    def _convert(val):
        if pd.isna(val):
            return val
        s = str(val).strip()
        if decimal_comma_re.match(s):
            return float(s.replace(",", "."))
        try:
            return float(s)
        except (ValueError, TypeError):
            return val

    converted = series.apply(_convert)
    try:
        converted = pd.to_numeric(converted)
    except (ValueError, TypeError):
        pass
    return converted, True


# Columns that may contain decimal commas. Excluded by design:
#   - `dr` (diastereomeric ratio, e.g. "25:1")
#   - `time` if it may contain "24 h" / "overnight"
_numeric_candidates = [
    "ee", "yield", "temperature",
    "imine/enol", "imine_enol",
    "catalyst_concentration", "catalyst_con",
    "acid_additive_equiv", "base_additive_equiv", "additives_equiv",
    "water_additive", "water_additive_equiv",
]

conversions = []
for col in _numeric_candidates:
    if col in df.columns:
        converted, changed = _fix_comma_decimal(df[col])
        df[col] = converted
        if changed:
            conversions.append(col)

if conversions:
    print(f"Converted decimal commas to dots in: {conversions}")


# Normalize axial_configuration
def _normalize_axial(v):
    if not _is_filled(v):
        return "none"
    s = str(v).strip().upper()
    return s if s in ("R", "S") else "none"


df[COL_AXIAL] = df[COL_AXIAL].apply(_normalize_axial)
print(f"\nAxial configuration distribution:")
print(df[COL_AXIAL].value_counts())

In [ ]:
def make_catalyst_id(smiles, axial):
    """Construct a unique catalyst identifier from the full InChIKey.

    The InChIKey has the form ``AAAAAAAAAAAAAA-BBBBBBBBFV-P`` where the first
    block encodes connectivity and the second block encodes stereochemistry.
    Using both blocks (plus the axial configuration) guarantees that
    enantiomers and diastereomers receive distinct IDs.
    """
    if not _is_filled(smiles):
        return "INVALID"
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return "INVALID"
    ikey = Chem.MolToInchiKey(mol)
    axial_str = str(axial).strip().upper() if _is_filled(axial) else "NONE"
    parts = ikey.split("-")
    if len(parts) >= 2:
        return f"{parts[0]}_{parts[1]}_{axial_str}"
    return f"{parts[0]}_{axial_str}"


df["catalyst_inchikey"] = df[COL_SMILES].apply(
    lambda s: Chem.MolToInchiKey(Chem.MolFromSmiles(s))
              if _is_filled(s) and Chem.MolFromSmiles(s) else None
)
df["catalyst_id"] = df.apply(
    lambda r: make_catalyst_id(r[COL_SMILES], r[COL_AXIAL]), axis=1
)

# Assign sequential filenames: catalyst1.sdf, catalyst2.sdf, ...
unique_ids = df.drop_duplicates("catalyst_id").sort_values("catalyst_id").reset_index(drop=True)
filename_map = {row["catalyst_id"]: f"{CATALYST_PREFIX}{i + 1}"
                for i, row in unique_ids.iterrows()}
df["catalyst_filename"] = df["catalyst_id"].map(filename_map)

print(f"Total reactions:      {len(df)}")
print(f"Unique catalysts:     {df['catalyst_id'].nunique()}")
df[[COL_SMILES, COL_AXIAL, "catalyst_id", "catalyst_filename"]].drop_duplicates(
    "catalyst_id"
).head(10)

## 5. Calculate ΔΔG‡ from ee

The enantiomeric excess is related to the difference between the two competing transition state free energies through the Eyring relation:

$$|\Delta\Delta G^{\ddagger}| = -RT \ln\!\left(\frac{1 + ee/100}{1 - ee/100}\right)$$

where $R$ = 1.987 × 10⁻³ kcal mol⁻¹ K⁻¹, $T$ is the reaction temperature in Kelvin, and $ee$ is the enantiomeric excess in percent.

In this dataset $ee$ is recorded as a positive number (the directional information about the major enantiomer is encoded in the SMILES of the major product, stored in a separate column). The resulting `ddG_kcal_mol` column therefore stores the **magnitude** of ΔΔG‡ in kcal mol⁻¹. A signed version, if required for machine learning, can be derived downstream by comparing the major product SMILES to a reference scaffold.

In [ ]:
R_KCAL = 1.987e-3  # kcal/(mol K)


def compute_ddg(ee_pct, temperature, temp_unit="celsius", ee_cap=EE_UPPER_CAP_PCT):
    """Return ``|ΔΔG‡|`` in kcal/mol and a status note."""
    if not _is_filled(ee_pct):
        return np.nan, "ee_missing"
    try:
        ee = float(ee_pct)
    except (ValueError, TypeError):
        return np.nan, "ee_invalid"
    if ee < 0:
        return np.nan, "ee_negative"
    if ee > 100:
        return np.nan, "ee_above_100"
    if ee >= ee_cap:
        ee = ee_cap

    if not _is_filled(temperature):
        return np.nan, "temp_missing"
    try:
        t_raw = float(temperature)
    except (ValueError, TypeError):
        return np.nan, "temp_invalid"

    if temp_unit == "celsius":
        t_kelvin = t_raw + 273.15
    elif temp_unit == "kelvin":
        t_kelvin = t_raw
    else:
        return np.nan, f"unknown_temp_unit:{temp_unit}"
    if t_kelvin <= 0:
        return np.nan, "temp_non_positive"

    try:
        ratio = (1 + ee / 100) / (1 - ee / 100)
        ddg = -R_KCAL * t_kelvin * np.log(ratio)
        return abs(ddg), "ok"
    except Exception as exc:
        return np.nan, f"calc_error:{exc}"


if COMPUTE_DDG:
    missing = [c for c in [COL_EE, COL_TEMPERATURE] if c not in df.columns]
    if missing:
        print(f"Skipped ΔΔG‡ computation; missing columns: {missing}")
        df["ddG_kcal_mol"] = np.nan
        df["ddG_calc_note"] = "missing_columns"
    else:
        results = [compute_ddg(r[COL_EE], r[COL_TEMPERATURE], TEMPERATURE_UNIT)
                   for _, r in df.iterrows()]
        df["ddG_kcal_mol"] = [r[0] for r in results]
        df["ddG_calc_note"] = [r[1] for r in results]

        ok = df[df["ddG_calc_note"] == "ok"]
        print(f"ΔΔG‡ computed for {len(ok)} / {len(df)} rows")
        if len(ok):
            print(f"\n|ΔΔG‡| statistics (kcal/mol):")
            print(f"  min:    {ok['ddG_kcal_mol'].min():.3f}")
            print(f"  median: {ok['ddG_kcal_mol'].median():.3f}")
            print(f"  mean:   {ok['ddG_kcal_mol'].mean():.3f}")
            print(f"  max:    {ok['ddG_kcal_mol'].max():.3f}")
else:
    print("ΔΔG‡ computation disabled.")

## 6. Pre-flight validation of catalyst SMILES

Each unique catalyst SMILES is examined before 3D generation to identify potential issues: invalid SMILES, undefined stereocenters (excluding bridgehead atoms in bridged bicyclic systems, which RDKit reports as undefined for legitimate topological reasons), and catalysts whose `axial_configuration` is set but for which no biaryl axis can be detected.

In [ ]:
def _bridgehead_atoms(mol):
    """Return the set of atom indices that are part of two or more rings."""
    ri = mol.GetRingInfo()
    ring_count = {}
    for ring in ri.AtomRings():
        for a in ring:
            ring_count[a] = ring_count.get(a, 0) + 1
    return {a for a, n in ring_count.items() if n >= 2}


def preflight(df):
    """Run pre-flight SMILES checks and tally strategy assignments."""
    issues = []
    strategy_count = {"plain": 0, "free_biaryl": 0, "bridged_biaryl": 0,
                      "plain_no_axis": 0, "invalid": 0}

    for _, row in df.drop_duplicates("catalyst_id").iterrows():
        cid, smi, axial = row["catalyst_id"], row[COL_SMILES], row[COL_AXIAL]

        if not _is_filled(smi):
            issues.append({"catalyst_id": cid, "severity": "critical",
                           "issue": "empty SMILES"})
            strategy_count["invalid"] += 1
            continue

        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            issues.append({"catalyst_id": cid, "severity": "critical",
                           "issue": "RDKit cannot parse SMILES"})
            strategy_count["invalid"] += 1
            continue

        bridgeheads = _bridgehead_atoms(mol)
        unspec = [c for c in Chem.FindMolChiralCenters(
            mol, includeUnassigned=True, useLegacyImplementation=False)
            if c[1] == "?" and c[0] not in bridgeheads]
        if unspec:
            issues.append({"catalyst_id": cid, "severity": "warning",
                           "issue": f"{len(unspec)} undefined stereocenter(s)"})

        mol_h = Chem.AddHs(mol)
        axis = find_biaryl_axis(mol_h)
        needs_axial = _is_filled(axial) and str(axial).strip().upper() in ("R", "S")

        if not needs_axial:
            strategy_count["plain"] += 1
        elif axis is None:
            issues.append({"catalyst_id": cid, "severity": "warning",
                           "issue": "axial set but no biaryl axis detected"})
            strategy_count["plain_no_axis"] += 1
        elif _bond_in_ring(mol_h, axis[0], axis[1]):
            strategy_count["bridged_biaryl"] += 1
        else:
            strategy_count["free_biaryl"] += 1

    return pd.DataFrame(issues), strategy_count


if DO_PREFLIGHT_CHECK:
    issues_df, strat = preflight(df)

    print("Strategy distribution:")
    for s, n in strat.items():
        if n > 0:
            print(f"  {s:20s}  {n}")

    if len(issues_df) == 0:
        print("\nNo issues detected.")
    else:
        print(f"\nIssues found: {len(issues_df)}")
        for sev in ("critical", "warning"):
            subset = issues_df[issues_df["severity"] == sev]
            if len(subset):
                print(f"\n{sev.upper()} ({len(subset)}):")
                print(subset.drop(columns="severity").to_string(index=False))
else:
    print("Pre-flight check skipped.")

## 7. Generate 3D structures and write SDF files

Each unique catalyst is processed once. The selected 3D conformer is written to an individual SDF file in MOL V3000 format with provenance metadata in the SDF tags.

In [ ]:
def write_sdf(mol, path, catalyst_id, source_smiles, axial, build_source,
              filename, inchikey="", final_angle=None, n_valid_conformers=None):
    """Write a 3D structure to a V3000 SDF file with provenance tags."""
    mol.SetProp("_Name", filename)
    mol.SetProp("catalyst_id", catalyst_id)
    mol.SetProp("source_smiles", str(source_smiles))
    mol.SetProp("axial_configuration", str(axial))
    mol.SetProp("inchikey", str(inchikey))
    mol.SetProp("build_source", build_source)
    if final_angle is not None:
        mol.SetProp("final_biaryl_angle_deg", f"{final_angle:.2f}")
    if n_valid_conformers is not None:
        mol.SetProp("n_valid_conformers", str(n_valid_conformers))
    mol.SetProp("rdkit_version", Chem.rdBase.rdkitVersion)
    mol.SetProp("random_seed", str(RANDOM_SEED))
    mol.SetProp("generated_at", datetime.now().isoformat(timespec="seconds"))

    writer = Chem.SDWriter(str(path))
    writer.SetKekulize(True)
    writer.SetForceV3000(True)
    writer.write(mol)
    writer.close()


unique_catalysts = df.drop_duplicates("catalyst_id").reset_index(drop=True)
print(f"Generating 3D structures for {len(unique_catalysts)} unique catalysts...")

results = []
for _, row in tqdm(unique_catalysts.iterrows(),
                   total=len(unique_catalysts),
                   desc="Building SDFs"):
    cid      = row["catalyst_id"]
    smi      = row[COL_SMILES]
    axial    = row[COL_AXIAL]
    inchikey = row.get("catalyst_inchikey", "")
    fname    = row["catalyst_filename"]

    try:
        mol, source, info = build_3d_catalyst(smi, axial_config=axial)
    except Exception as exc:
        mol, source, info = None, f"exception:{exc}", {}

    record = {
        "catalyst_id":         cid,
        "catalyst_filename":   fname,
        "smiles":              smi,
        "axial_configuration": axial,
        "inchikey":            inchikey,
        "build_source":        source,
        "strategy":            info.get("strategy", ""),
        "n_valid_conformers":  info.get("n_valid_conformers"),
    }

    if mol is None:
        record.update({"status": "failed", "sdf_path": "",
                       "structure_match": None, "axial_match": None,
                       "axial_angle": None, "n_point_stereocenters": None,
                       "n_rotatable_bonds": None})
        results.append(record)
        continue

    sdf_path = Path(OUTPUT_SDF_DIR) / f"{fname}.sdf"
    try:
        write_sdf(mol, sdf_path, cid, smi, axial, source, fname, inchikey,
                  final_angle=info.get("final_angle"),
                  n_valid_conformers=info.get("n_valid_conformers"))
        record["status"]   = "ok"
        record["sdf_path"] = str(sdf_path)
    except Exception as exc:
        record["status"]   = f"sdf_write_failed:{exc}"
        record["sdf_path"] = ""

    val = validate_3d_structure(mol, expected_axial=axial, expected_smiles=smi, info=info)
    record.update(val)

    try:
        record["n_rotatable_bonds"] = Lipinski.NumRotatableBonds(Chem.MolFromSmiles(smi))
    except Exception:
        record["n_rotatable_bonds"] = None

    results.append(record)

build_report = pd.DataFrame(results)
ok_count     = (build_report["status"] == "ok").sum()
print(f"\nSuccessful: {ok_count} / {len(build_report)}")

In [ ]:
# Summary
print("Strategy assignments:")
print(build_report["strategy"].value_counts().to_string())

ok = build_report[build_report["status"] == "ok"]
print(f"\nValidation of successful structures:")
print(f"  structure_match:  {ok['structure_match'].sum()} / {len(ok)}")

axial_subset = ok[ok["axial_configuration"].isin(["R", "S"])]
if len(axial_subset) > 0:
    matched = axial_subset["axial_match"].fillna(False).sum()
    print(f"  axial_match:      {matched} / {len(axial_subset)}")
    print(f"  mean |angle|:     {axial_subset['axial_angle'].abs().mean():.1f} deg")

failed = build_report[build_report["status"] != "ok"]
if len(failed):
    print(f"\nFailed structures: {len(failed)}")
    print(failed[["catalyst_filename", "smiles", "axial_configuration",
                  "strategy", "status"]].to_string(index=False))

## 8. Stereochemical diagnostics for enantiomeric pairs

For each scaffold present as both R and S atropoisomers, the RMSD of direct alignment and of alignment after inversion through the geometric center is computed. True enantiomers have a large direct RMSD (the two structures are non-superimposable) and a near-zero mirror-aligned RMSD (mirror images are superimposable).

**Visualization note.** When R and S enantiomers are overlaid in PyMOL or ChimeraX with the `align` command, they may appear identical. This is the expected behavior: the alignment algorithm permits a mirror reflection if it minimizes RMSD. To observe the actual structural difference, load both files **without** alignment.

In [ ]:
ok = build_report[build_report["status"] == "ok"].copy()
atropos = ok[ok["axial_configuration"].isin(["R", "S"])].copy()

if len(atropos) > 0:
    atropos["scaffold_key"] = atropos["catalyst_id"].str.split("_").str[0]
    pairs = atropos.groupby("scaffold_key").agg(
        configs=("axial_configuration", list),
        angles=("axial_angle", list),
        filenames=("catalyst_filename", list),
        paths=("sdf_path", list),
    ).reset_index()

    enantio_pairs = pairs[pairs["configs"].apply(lambda x: set(x) >= {"R", "S"})]
    print(f"Enantiomeric R/S pairs found: {len(enantio_pairs)}\n")

    for _, row in enantio_pairs.iterrows():
        print(f"Scaffold {row['scaffold_key']}:")
        for cfg, ang, fname in zip(row["configs"], row["angles"], row["filenames"]):
            sign_ok = ((cfg == "R" and ang and ang > 0) or
                       (cfg == "S" and ang and ang < 0))
            mark = "OK" if sign_ok else "??"
            ang_str = f"{ang:+.1f} deg" if ang else "n/a"
            print(f"  {fname:18s}  axial={cfg}  angle={ang_str}  [{mark}]")

        try:
            r_idx = row["configs"].index("R")
            s_idx = row["configs"].index("S")
            mol_r = Chem.SDMolSupplier(row["paths"][r_idx], removeHs=False)[0]
            mol_s = Chem.SDMolSupplier(row["paths"][s_idx], removeHs=False)[0]

            mol_s_inv = Chem.Mol(mol_s)
            conf = mol_s_inv.GetConformer()
            coords = np.array([list(conf.GetAtomPosition(j))
                               for j in range(mol_s_inv.GetNumAtoms())])
            center = coords.mean(axis=0)
            for j in range(mol_s_inv.GetNumAtoms()):
                conf.SetAtomPosition(j, tuple(2 * center - coords[j]))

            rmsd_direct = rdMolAlign.GetBestRMS(mol_r, mol_s)
            rmsd_mirror = rdMolAlign.GetBestRMS(mol_r, mol_s_inv)
            print(f"  RMSD(direct):     {rmsd_direct:.3f} A")
            print(f"  RMSD(mirrored S): {rmsd_mirror:.3f} A")
            verdict = "TRUE_ENANTIOMERS" if rmsd_mirror < 0.5 and rmsd_direct > 1.0 \
                      else "CHECK_MANUALLY"
            print(f"  Verdict: {verdict}\n")
        except Exception as exc:
            print(f"  RMSD calculation failed: {exc}\n")
else:
    print("No atropoisomers with explicit R/S in the dataset.")

## 8b. Verification of axial chirality against 3D coordinates

For atropoisomeric catalysts, the assigned axial configuration (R/S) is verified directly against the generated 3D coordinates. The biaryl dihedral angle is measured from the atom positions, and its sign is checked for internal consistency: all catalysts sharing a scaffold and an axial label must give the same dihedral sign, and the R and S forms of a scaffold must give opposite signs.

A strict Cahn–Ingold–Prelog assignment is also attempted via `rdCIPLabeler`. Note that RDKit does not reliably perceive atropisomerism from 3D coordinates alone, so the resulting CIP column is frequently empty for biaryls; this is a known limitation and is not an error. The dihedral-sign consistency is therefore the authoritative, coordinate-based verification, while the measured angle provides an objective geometric descriptor independent of any label.

In [ ]:
from rdkit.Chem import rdCIPLabeler


def _cip_label_from_3d(mol):
    """Attempt strict CIP labelling; return the first bond CIP code or 'none'."""
    Chem.AssignStereochemistryFrom3D(mol)
    try:
        rdCIPLabeler.AssignCIPLabels(mol)
    except Exception:
        pass
    for bond in mol.GetBonds():
        if bond.HasProp("_CIPCode"):
            return bond.GetProp("_CIPCode")
    return "none"


# Collect atropoisomers from the generated SDF files
verify_records = []
for _, row in build_report[build_report["status"] == "ok"].iterrows():
    if row["axial_configuration"] not in ("R", "S"):
        continue
    mol = Chem.SDMolSupplier(row["sdf_path"], removeHs=False)[0]
    if mol is None:
        continue
    axis = find_biaryl_axis(mol)
    angle = _measure_dihedral(mol, axis) if axis else None
    verify_records.append({
        "catalyst_filename": row["catalyst_filename"],
        "scaffold": row["catalyst_id"].split("_")[0],
        "axial": row["axial_configuration"],
        "angle_deg": round(angle, 2) if angle is not None else None,
        "sign": ("+" if angle > 0 else "-") if angle is not None else None,
        "cip_from_3d": _cip_label_from_3d(mol),
    })

if not verify_records:
    print("No atropoisomers with explicit R/S configuration in the dataset.")
else:
    verify_df = pd.DataFrame(verify_records)

    # Consistency check
    issues = []
    for scaffold, grp in verify_df.groupby("scaffold"):
        sign_by_axial = {}
        for axial in ("R", "S"):
            signs = set(grp[grp["axial"] == axial]["sign"].dropna())
            if signs:
                sign_by_axial[axial] = signs
                if len(signs) > 1:
                    issues.append(f"Scaffold {scaffold}: axial={axial} "
                                  f"has inconsistent signs {signs}")
        if "R" in sign_by_axial and "S" in sign_by_axial:
            if (len(sign_by_axial["R"]) == 1 and len(sign_by_axial["S"]) == 1
                    and sign_by_axial["R"] == sign_by_axial["S"]):
                issues.append(f"Scaffold {scaffold}: R and S share the same sign "
                              f"(expected opposite)")

    n_atropos = len(verify_df)
    n_cip = (verify_df["cip_from_3d"] != "none").sum()
    print(f"Atropoisomers verified: {n_atropos}")
    print(f"rdCIPLabeler bond CIP assigned: {n_cip} / {n_atropos} "
          f"(empty is expected for biaryls)")
    print(f"Mean |dihedral|: {verify_df['angle_deg'].abs().mean():.1f} deg\n")

    if not issues:
        print("Consistency check PASSED: R and S have opposite, internally")
        print("consistent dihedral signs across all scaffolds.")
    else:
        print("Consistency check found issues:")
        for issue in issues:
            print(f"  - {issue}")

    verify_df

## 9. Export enriched dataset and SDF archive

Two enriched versions of the dataset are written:

* **`*_full`** — includes all auxiliary columns useful for inspection and downstream debugging.
* **`*_minimal`** — contains only the original reaction columns plus `ddG_kcal_mol` (the ML target) and `catalyst_3d_sdf` (the link to the 3D structure file), and is the recommended input for descriptor calculation pipelines.

The `catalyst_mapping.csv` table connects each numbered SDF filename to the underlying SMILES and InChIKey. All SDF files are bundled into a ZIP archive together with a build report and a README for supplementary materials.

In [ ]:
# Map catalyst_id to its SDF path
id_to_sdf    = dict(zip(build_report["catalyst_id"], build_report["sdf_path"]))
id_to_source = dict(zip(build_report["catalyst_id"], build_report["build_source"]))
id_to_status = dict(zip(build_report["catalyst_id"], build_report["status"]))

df["catalyst_3d_sdf"]       = df["catalyst_id"].map(id_to_sdf)
df["catalyst_build_source"] = df["catalyst_id"].map(id_to_source)
df["catalyst_build_status"] = df["catalyst_id"].map(id_to_status)


def save_dataset(dataframe, path):
    if Path(path).suffix.lower() in (".xlsx", ".xls"):
        dataframe.to_excel(path, index=False, sheet_name="data")
    else:
        dataframe.to_csv(path, index=False)


# Save the catalyst mapping
mapping_cols = ["catalyst_filename", "catalyst_id", "smiles", "axial_configuration",
                "inchikey", "strategy", "build_source", "status",
                "n_rotatable_bonds", "axial_angle"]
mapping_cols = [c for c in mapping_cols if c in build_report.columns]
mapping_df = (build_report[mapping_cols]
              .sort_values("catalyst_filename")
              .reset_index(drop=True))
mapping_df.to_csv(CATALYST_MAPPING_CSV, index=False)
print(f"Mapping saved: {CATALYST_MAPPING_CSV}")

# Full enriched dataset
save_dataset(df, OUTPUT_ENRICHED_FULL)
print(f"Full dataset:     {OUTPUT_ENRICHED_FULL}  ({len(df.columns)} columns)")

# Minimal dataset
_drop_cols = [c for c in ["catalyst_inchikey", "catalyst_id", "catalyst_filename",
                          "catalyst_build_source", "catalyst_build_status",
                          "ddG_calc_note"]
              if c in df.columns]
df_minimal = df.drop(columns=_drop_cols)
save_dataset(df_minimal, OUTPUT_ENRICHED_MINIMAL)
print(f"Minimal dataset:  {OUTPUT_ENRICHED_MINIMAL}  ({len(df_minimal.columns)} columns)")

In [ ]:
# Save the build report alongside the SDF files
report_path = Path(OUTPUT_SDF_DIR) / "_build_report.csv"
build_report.to_csv(report_path, index=False)
shutil.copy(CATALYST_MAPPING_CSV, Path(OUTPUT_SDF_DIR) / "catalyst_mapping.csv")

# Generate a README for the supplementary archive
readme = f"""# 3D structures of organocatalysts

This archive contains 3D conformers of all unique organocatalysts present in
the asymmetric Mannich reaction dataset.

## File naming

Each SDF file corresponds to a single unique catalyst. The mapping between
filenames and chemical structures is given in `catalyst_mapping.csv`.

## Generation details

- RDKit version:      {Chem.rdBase.rdkitVersion}
- Random seed:        {RANDOM_SEED}
- Force field:        MMFF94
- N conformers:       {N_CONFORMERS} (plain & free biaryl)
- N seeds (bridged):  {N_BRIDGED_SEEDS}
- SDF format:         MOL V3000
- Generation date:    {datetime.now().isoformat(timespec="seconds")}

## Strategies for 3D generation

1. plain                 - Standard ETKDGv3 embedding with enforceChirality=True
                           and MMFF94 minimization. Used for catalysts without
                           a biaryl axis (prolines, MacMillan, thioureas,
                           cinchona alkaloids).

2. smiles+axial_free     - For free biaryl atropoisomers (classical BINOL).
                           The biaryl dihedral is set to +90 (R) or -90 (S)
                           and held within +/- 5 degrees by an MMFF torsion
                           constraint (force constant 1.0e4).

3. smiles+axial_bridged  - For bridged biaryl atropoisomers (NOBIN-type).
                           Multiple seeds are tested; conformers with the
                           correct dihedral sign are retained and the
                           lowest-energy one is selected. Typical dihedrals
                           are ~+/- 120 degrees, set by ring geometry.

## SDF metadata fields

Each SDF file contains the following property tags:

- catalyst_id              Unique identifier (full InChIKey + axial)
- source_smiles            SMILES from the curated dataset
- axial_configuration      R / S / none
- inchikey                 Standard InChIKey
- build_source             Strategy applied
- final_biaryl_angle_deg   Dihedral of the biaryl axis (atropoisomers only)
- n_valid_conformers       Number of valid conformers (bridged biaryls only)
- rdkit_version            For reproducibility
- random_seed              For reproducibility
- generated_at             ISO timestamp

## R/S visualization

When R and S enantiomers are overlaid in PyMOL or ChimeraX using the `align`
command, they may appear identical. This is a property of the alignment
algorithm, which allows mirror reflection to minimize RMSD. To inspect the
true structural difference, load both structures WITHOUT alignment.

For true enantiomers:
  RMSD(direct)              > 1 A     (the two structures differ in 3D)
  RMSD(after inversion of S) ~ 0 A    (R and inverted S are superimposable)

## Files

- catalyst*.sdf             3D structures (one per unique catalyst)
- catalyst_mapping.csv      Filename to structure correspondence
- _build_report.csv         Full validation and diagnostic report
- README.txt                This file
"""
(Path(OUTPUT_SDF_DIR) / "README.txt").write_text(readme, encoding="utf-8")

# Create the ZIP archive
with zipfile.ZipFile(OUTPUT_ARCHIVE, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    n = 0
    for f in sorted(Path(OUTPUT_SDF_DIR).iterdir()):
        if f.is_file():
            zf.write(f, arcname=Path(OUTPUT_SDF_DIR) / f.name)
            n += 1

size_mb = Path(OUTPUT_ARCHIVE).stat().st_size / 1024 ** 2
print(f"Archive: {OUTPUT_ARCHIVE}  ({n} files, {size_mb:.2f} MB)")

## 10. Summary

The notebook produced:

* `<input>_full.{csv,xlsx}` — the dataset augmented with `ddG_kcal_mol`, `catalyst_id`, `catalyst_filename`, `catalyst_3d_sdf`, and provenance columns.
* `<input>_minimal.{csv,xlsx}` — a clean machine-learning-ready table containing only the original columns plus `ddG_kcal_mol` and `catalyst_3d_sdf`.
* `catalyst_mapping.csv` — filename-to-structure correspondence.
* `catalysts_3d/` — directory of MOL V3000 SDF files (one per unique catalyst).
* `catalysts_3d.zip` — archived supplementary material ready for journal submission.

The SDF structures are suitable as input for steric descriptor calculation (Sterimol, %V<sub>bur</sub>, MORFEUS-derived buried volumes), pharmacophore descriptor extraction (pmapper), and graph-based learning models (DMPNN, GNN). The `ddG_kcal_mol` column provides the target variable for regression models predicting enantioselectivity.

---

## References

1. Riniker, S.; Landrum, G. A. Better Informed Distance Geometry: Using What We Know To Improve Conformation Generation. *J. Chem. Inf. Model.* **2015**, *55*, 2562–2574.
2. Halgren, T. A. Merck Molecular Force Field. I. Basis, Form, Scope, Parameterization, and Performance of MMFF94. *J. Comput. Chem.* **1996**, *17*, 490–519.
3. Heller, S. R.; McNaught, A.; Pletnev, I.; Stein, S.; Tchekhovskoi, D. InChI, the IUPAC International Chemical Identifier. *J. Cheminform.* **2015**, *7*, 23.
4. Landrum, G. RDKit: Open-source cheminformatics. https://www.rdkit.org